# Create survey-disjoint beach-surface splits

All labelled cells from 20180812, 20180915, and 20220910 are reserved for testing. Surveys 20230731 and 20240623 are reserved for validation. Every other survey is used for training. All three sets must contain all ten classes. Targets and context neighbours always come from the same survey, so their input observations cannot cross split boundaries.

Only annotation metadata is written to Data/Preprocessed; matrix files are loaded from Data/Raw/beach_all. Historical outputs below require rerunning.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    raise RuntimeError(
        "Could not locate project root containing Source and Notebooks."
    )

DESCRIPTOR_PATH = PROJECT_ROOT / "Data" / "Raw" / "Descriptor_fs.csv"
CELL_DATA_ROOT = PROJECT_ROOT / "Data" / "Raw" / "beach_all"
PREPROCESSED_DIR = PROJECT_ROOT / "Data" / "Preprocessed"
FEATURE_SUFFIXES = ("r", "g", "b", "s", "d")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from Source.bs_splits import (
    TEST_DATES, VALIDATION_DATES, assign_bs_splits, save_bs_split_frames,
)

print(f"Project root:     {PROJECT_ROOT}")
print(f"Descriptor:       {DESCRIPTOR_PATH}")
print(f"Cell data root:   {CELL_DATA_ROOT}")
print(f"Output directory: {PREPROCESSED_DIR}")

In [ ]:
DESCRIPTOR_COLUMNS = ["date", "cell_id", "class"]
DESCRIPTOR_DTYPES = {column: "string" for column in DESCRIPTOR_COLUMNS}


def read_descriptor(path):
    if not path.exists():
        raise FileNotFoundError(f"Descriptor file not found: {path}")
    frame = pd.read_csv(
        path,
        sep=";",
        header=None,
        names=DESCRIPTOR_COLUMNS,
        dtype=DESCRIPTOR_DTYPES,
        encoding="utf-8-sig",
    )

    frame = frame.dropna(subset=DESCRIPTOR_COLUMNS).copy()
    for column in DESCRIPTOR_COLUMNS:
        frame[column] = frame[column].str.strip()
    empty_mask = frame[DESCRIPTOR_COLUMNS].eq("").any(axis=1)
    if empty_mask.any():
        raise ValueError(f"Empty descriptor values found in {path}: {int(empty_mask.sum())} rows")
    return frame


descriptor_raw = read_descriptor(DESCRIPTOR_PATH)

duplicate_mask = descriptor_raw.duplicated(subset=["date", "cell_id"], keep=False)
if duplicate_mask.any():
    duplicates = descriptor_raw.loc[duplicate_mask].sort_values(["date", "cell_id", "class"])
    conflicting_keys = (
        duplicates.groupby(["date", "cell_id"])["class"].nunique().loc[lambda counts: counts > 1]
    )
    if not conflicting_keys.empty:
        conflicts = duplicates.merge(
            conflicting_keys.rename("class_count"),
            on=["date", "cell_id"],
        )
        display(conflicts)
        raise ValueError("Conflicting classes found for duplicate date/cell_id pairs.")
    duplicate_key_count = duplicates[["date", "cell_id"]].drop_duplicates().shape[0]
    print(f"Dropping {len(duplicates) - duplicate_key_count:,} repeated annotations.")

descriptor = (
    descriptor_raw
    .drop_duplicates(subset=["date", "cell_id"], keep="first")
    .sort_values(["date", "cell_id"])
    .reset_index(drop=True)
)
descriptor.head(), descriptor.shape

In [ ]:
if not CELL_DATA_ROOT.exists():
    raise FileNotFoundError(f"Cell data root not found: {CELL_DATA_ROOT}")

missing_rows = []
for row in descriptor.itertuples(index=False):
    missing_features = [
        suffix
        for suffix in FEATURE_SUFFIXES
        if not (CELL_DATA_ROOT / row.date / f"{row.cell_id}{suffix}.txt").is_file()
    ]
    if missing_features:
        missing_rows.append(
            {
                "date": row.date,
                "cell_id": row.cell_id,
                "class": row[2],
                "missing_features": ",".join(missing_features),
            }
        )

missing_cells = pd.DataFrame(missing_rows)
if not missing_cells.empty:
    display(missing_cells.head(50))
    raise FileNotFoundError(
        f"{len(missing_cells):,} annotated cells are incomplete under {CELL_DATA_ROOT}."
    )

print(f"Validated five feature files for all {len(descriptor):,} annotated cells in {CELL_DATA_ROOT}.")

class_counts = descriptor["class"].value_counts().sort_index()
class_counts.to_frame(name="count")

In [ ]:
train_df, validation_df, test_df = assign_bs_splits(descriptor)
split_df = pd.concat([train_df, validation_df, test_df], ignore_index=True)
split_df = split_df.sort_values(["split", "date", "cell_id"]).reset_index(drop=True)
display(split_df.groupby("split").size().rename("cells"))
display(split_df.groupby("split")["date"].unique())

In [ ]:
expected_keys = set(map(tuple, descriptor[["date", "cell_id"]].to_numpy()))
actual_keys = set(map(tuple, split_df[["date", "cell_id"]].to_numpy()))

if expected_keys != actual_keys:
    missing = expected_keys - actual_keys
    unexpected = actual_keys - expected_keys
    raise AssertionError(f"Split mismatch. Missing={len(missing)}, unexpected={len(unexpected)}")

if split_df.duplicated(subset=["date", "cell_id"]).any():
    raise AssertionError("A date/cell_id pair appears in more than one split.")

distribution = (
    split_df
    .groupby(["split", "class"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(["train", "validation", "test"])
)

display(distribution)
assert set(test_df["date"]) == TEST_DATES
assert not train_df["date"].isin(TEST_DATES).any()
assert not validation_df["date"].isin(TEST_DATES).any()
assert len(test_df) == int(descriptor["date"].isin(TEST_DATES).sum())
assert set(split_df["class"]) == set(descriptor["class"])

assert set(validation_df["date"]) == VALIDATION_DATES
assert not train_df["date"].isin(VALIDATION_DATES).any()
assert (distribution > 0).all().all()

In [ ]:
output_files, split_provenance = save_bs_split_frames(
    train_df, validation_df, test_df, PREPROCESSED_DIR,
)
for split_name, path in output_files.items():
    print(f"{split_name:>10}: {path}", flush=True)
display(pd.DataFrame([split_provenance]))